# 04 Circuit Analysis

This notebook analyzes circuit-level characteristics: difficulty, lap-time evolution, overtaking frequency, weather context, and approximate circuit speed profile.

Telemetry is sampled from Parquet row groups only where needed; the notebook does not load full telemetry into memory.

In [1]:
from pathlib import Path
import sys
from datetime import datetime
import json

import pandas as pd
import numpy as np
import plotly.express as px
import pyarrow.parquet as pq

ROOT = Path.cwd()
while not (ROOT / "configs" / "pipeline_config.yaml").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent

SHARED = ROOT / "eda" / "shared" / "scripts"
if str(SHARED) not in sys.path:
    sys.path.insert(0, str(SHARED))

from config import CLEANED_DATA_PATH

NOTEBOOK_NAME = "04_circuit_analysis"
OUTPUT_TABLES = ROOT / "eda" / "silver" / "outputs" / "tables" / NOTEBOOK_NAME
OUTPUT_CHARTS = ROOT / "eda" / "silver" / "outputs" / "charts" / NOTEBOOK_NAME
OUTPUT_REPORTS = ROOT / "eda" / "silver" / "outputs" / "reports" / NOTEBOOK_NAME
INSIGHTS = ROOT / "eda" / "silver" / "insights"
CHECKPOINTS = ROOT / "eda" / "silver" / "checkpoints"
for path in [OUTPUT_TABLES, OUTPUT_CHARTS, OUTPUT_REPORTS, INSIGHTS, CHECKPOINTS]:
    path.mkdir(parents=True, exist_ok=True)

def write_report(name: str, payload: dict) -> None:
    (OUTPUT_REPORTS / f"{name}.json").write_text(json.dumps(payload, indent=2, default=str), encoding="utf-8")

def write_insight(title: str, observations: list[str], issues: list[str], recommendations: list[str]) -> None:
    content = f"# {title}\n\n"
    content += f"**Generated at:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n"
    content += "## Key Observations\n\n" + "\n".join(f"- {item}" for item in observations) + "\n\n"
    content += "## Issues\n\n" + ("\n".join(f"- {item}" for item in issues) if issues else "- None") + "\n\n"
    content += "## Recommendations\n\n" + "\n".join(f"- {item}" for item in recommendations) + "\n"
    (INSIGHTS / f"{NOTEBOOK_NAME}.md").write_text(content, encoding="utf-8")

print("=" * 72)
print(f"SILVER EDA - {NOTEBOOK_NAME}")
print(f"Start time: {datetime.now()}")
print(f"Cleaned data path: {CLEANED_DATA_PATH}")
print("=" * 72)


SILVER EDA - 04_circuit_analysis
Start time: 2026-06-02 00:46:14.979155
Cleaned data path: D:\F1_WinRate_Predictor\data\cleaned


In [2]:
import pyarrow.parquet as pq

session_result = pd.read_parquet(CLEANED_DATA_PATH / "session_result.parquet")
sessions = pd.read_parquet(CLEANED_DATA_PATH / "sessions.parquet")
meetings = pd.read_parquet(CLEANED_DATA_PATH / "meetings.parquet")
laps = pd.read_parquet(CLEANED_DATA_PATH / "laps.parquet")
overtakes = pd.read_parquet(CLEANED_DATA_PATH / "overtakes.parquet")
weather = pd.read_parquet(CLEANED_DATA_PATH / "weather.parquet")

sessions["event_type"] = np.where(
    sessions["session_name"].astype(str).str.lower().eq("sprint"),
    "SPRINT_RACE",
    "GRAND_PRIX_RACE",
)
race_info = sessions[["session_key", "meeting_key", "year", "event_type", "circuit_short_name", "country_name"]].drop_duplicates()
print(f"Sessions: {race_info['session_key'].nunique()}")
print(f"Circuits: {race_info['circuit_short_name'].nunique()}")
print(f"Countries: {race_info['country_name'].nunique()}")

Sessions: 70
Circuits: 24
Countries: 21


## 1. Circuit Difficulty

Circuit difficulty is approximated with DNF rate and finish-position completeness. This is not a pure track-only measure, but it flags venues where race outcomes are more failure-prone in the observed data.

In [3]:
result_circuit = session_result.merge(race_info, on=["session_key", "meeting_key"], how="left")
result_circuit["finish_pos"] = pd.to_numeric(result_circuit["position"], errors="coerce")
result_circuit["is_dnf_like"] = result_circuit[["dnf", "dns", "dsq"]].any(axis=1) | result_circuit["finish_pos"].isna()
circuit_difficulty = (
    result_circuit.groupby("circuit_short_name", as_index=False)
    .agg(
        entries=("driver_number", "count"),
        sessions=("session_key", "nunique"),
        dnf_rate=("is_dnf_like", "mean"),
        avg_finish=("finish_pos", "mean"),
        countries=("country_name", lambda s: ", ".join(sorted(set(s.dropna().astype(str))))),
    )
)
circuit_difficulty = circuit_difficulty[circuit_difficulty["entries"] >= 20].sort_values("dnf_rate", ascending=False)
circuit_difficulty.to_csv(OUTPUT_TABLES / "circuit_difficulty.csv", index=False)
display(circuit_difficulty.head(15))

,circuit_short_name,entries,sessions,dnf_rate,avg_finish,countries
9,Melbourne,61,3,0.245902,8.382979,Australia
13,Montreal,84,4,0.178571,9.400000,Canada
7,Las Vegas,40,2,0.175000,8.818182,United States
10,Mexico City,40,2,0.175000,9.000000,Mexico
17,Silverstone,40,2,0.175000,8.818182,United Kingdom
5,Interlagos,80,4,0.150000,9.188406,Brazil
12,Monte Carlo,40,2,0.150000,9.029412,Monaco
16,Shanghai,124,6,0.145161,9.523364,China
11,Miami,124,6,0.137097,9.448598,United States
1,Baku,40,2,0.125000,10.000000,Azerbaijan


In [4]:
fig = px.bar(
    circuit_difficulty.head(15).sort_values("dnf_rate"),
    x="dnf_rate",
    y="circuit_short_name",
    orientation="h",
    title="Circuit Difficulty Proxy: DNF-like Rate",
    labels={"dnf_rate": "DNF-like rate", "circuit_short_name": "Circuit"},
)
fig.write_html(OUTPUT_CHARTS / "circuit_difficulty.html", include_plotlyjs="cdn")
fig.show()

## 2. Lap-Time Evolution by Circuit

For recurring circuits, fastest observed lap time by year is used as a lightweight proxy for pace evolution. This is sensitive to weather, race type, and regulation changes, so it should be interpreted as EDA context.

In [5]:
lap_circuit = laps.merge(race_info[["session_key", "year", "circuit_short_name", "event_type"]], on="session_key", how="left")
lap_circuit["lap_duration"] = pd.to_numeric(lap_circuit["lap_duration"], errors="coerce")
valid_laps = lap_circuit[lap_circuit["lap_duration"].between(50, 900)].copy()
circuit_year = (
    valid_laps.groupby(["circuit_short_name", "year"], as_index=False)
    .agg(
        fastest_lap=("lap_duration", "min"),
        median_lap=("lap_duration", "median"),
        avg_lap=("lap_duration", "mean"),
        laps=("lap_duration", "count"),
    )
)
circuit_year = circuit_year.sort_values(["circuit_short_name", "year"])
circuit_year["prev_fastest_lap"] = circuit_year.groupby("circuit_short_name")["fastest_lap"].shift(1)
circuit_year["fastest_lap_improvement_pct"] = (circuit_year["prev_fastest_lap"] - circuit_year["fastest_lap"]) / circuit_year["prev_fastest_lap"] * 100
circuit_year.to_csv(OUTPUT_TABLES / "circuit_lap_time_trends.csv", index=False)
display(circuit_year.head(30))

,circuit_short_name,year,fastest_lap,median_lap,avg_lap,laps,prev_fastest_lap,fastest_lap_improvement_pct
0,Austin,2024,97.330,100.1160,102.394995,1435,NaN,NaN
1,Austin,2025,96.527,100.2655,105.893696,1380,97.330,0.825028
2,Baku,2024,105.255,109.0150,110.859709,968,NaN,NaN
3,Baku,2025,103.388,106.3555,111.009001,966,105.255,1.773787
4,Catalunya,2024,77.115,80.9080,81.470923,1309,NaN,NaN
5,Catalunya,2025,75.743,81.4560,84.699067,1201,77.115,1.779161
6,Hungaroring,2024,80.305,84.5215,85.142381,1354,NaN,NaN
7,Hungaroring,2025,79.409,82.4090,82.941545,1366,80.305,1.115746
8,Imola,2024,78.589,82.0690,82.751250,1236,NaN,NaN
9,Imola,2025,77.988,81.9600,87.356138,1205,78.589,0.764738


In [6]:
repeat_circuits = circuit_year.groupby("circuit_short_name")["year"].nunique()
selected = repeat_circuits[repeat_circuits >= 2].index[:10]
trend = circuit_year[circuit_year["circuit_short_name"].isin(selected)]
fig = px.line(
    trend,
    x="year",
    y="fastest_lap",
    color="circuit_short_name",
    markers=True,
    title="Fastest Lap Trend by Recurring Circuit",
)
fig.write_html(OUTPUT_CHARTS / "circuit_lap_time_trends.html", include_plotlyjs="cdn")
fig.show()

## 3. Overtaking Frequency

Overtaking volume is aggregated per circuit and normalized by session count. This helps identify tracks that generate more on-track position changes.

In [7]:
overtakes_circuit = overtakes.merge(race_info[["session_key", "year", "circuit_short_name", "event_type"]], on="session_key", how="left")
session_overtakes = overtakes_circuit.groupby(["session_key", "circuit_short_name"], as_index=False).size().rename(columns={"size": "overtakes"})
circuit_overtakes = (
    session_overtakes.groupby("circuit_short_name", as_index=False)
    .agg(avg_overtakes=("overtakes", "mean"), median_overtakes=("overtakes", "median"), sessions=("session_key", "nunique"))
    .sort_values("avg_overtakes", ascending=False)
)
circuit_overtakes.to_csv(OUTPUT_TABLES / "circuit_overtakes.csv", index=False)
display(circuit_overtakes.head(15))

,circuit_short_name,avg_overtakes,median_overtakes,sessions
15,Sakhir,527.500000,527.5,2
2,Catalunya,309.500000,309.5,2
22,Yas Marina Circuit,292.000000,292.0,2
23,Zandvoort,274.000000,274.0,2
10,Mexico City,263.000000,263.0,2
3,Hungaroring,246.500000,246.5,2
7,Las Vegas,245.000000,245.0,2
9,Melbourne,227.333333,211.0,3
18,Singapore,220.500000,220.5,2
8,Lusail,213.750000,219.5,4


In [8]:
fig = px.bar(
    circuit_overtakes.head(15).sort_values("avg_overtakes"),
    x="avg_overtakes",
    y="circuit_short_name",
    orientation="h",
    title="Average Overtakes by Circuit",
)
fig.write_html(OUTPUT_CHARTS / "circuit_overtakes.html", include_plotlyjs="cdn")
fig.show()

## 4. Circuit Weather Context

Weather is summarized by circuit to expose environmental differences that can later interact with tyre and pace features.

In [9]:
weather_circuit = weather.merge(race_info[["session_key", "circuit_short_name", "country_name"]], on="session_key", how="left")
weather_cols = [column for column in ["air_temperature", "track_temperature", "humidity", "pressure", "wind_speed", "rainfall"] if column in weather_circuit.columns]
circuit_weather = weather_circuit.groupby("circuit_short_name", as_index=False)[weather_cols].mean(numeric_only=True)
circuit_weather.to_csv(OUTPUT_TABLES / "circuit_weather.csv", index=False)
display(circuit_weather.sort_values("track_temperature", ascending=False).head(15))

,circuit_short_name,air_temperature,track_temperature,humidity,pressure,wind_speed,rainfall
20,Spielberg,29.557743,46.816798,36.257218,939.034383,1.518635,0.002625
14,Monza,29.805415,46.180144,37.649819,995.060650,1.748375,0.000000
12,Monte Carlo,21.871148,44.994958,58.901961,1017.874230,0.974790,0.000000
2,Catalunya,26.634304,44.831715,59.019417,1002.066343,1.877346,0.003236
0,Austin,28.270981,44.795198,37.298539,1002.294363,2.032985,0.000000
10,Mexico City,23.177570,43.165421,37.012461,1009.900000,1.997819,0.000000
4,Imola,24.525597,42.351536,43.245734,1008.254949,2.707850,0.000000
11,Miami,27.601346,40.075370,64.372005,1014.921534,1.902692,0.048452
3,Hungaroring,25.658076,38.899313,51.656357,984.224399,1.641237,0.000000
18,Singapore,29.930313,35.148125,74.512500,1009.403437,1.032500,0.018750


In [10]:
fig = px.scatter(
    circuit_weather,
    x="air_temperature",
    y="track_temperature",
    size="rainfall" if "rainfall" in circuit_weather.columns else None,
    hover_name="circuit_short_name",
    title="Circuit Weather Profile",
)
fig.write_html(OUTPUT_CHARTS / "circuit_weather_profile.html", include_plotlyjs="cdn")
fig.show()

## 5. Circuit Speed Profile from Telemetry Sample

To avoid memory pressure, speed profile uses a bounded sample from `car_data.parquet` row groups. This is enough for EDA-level circuit classification, not a final feature store.

In [11]:
def sample_car_data(max_row_groups: int = 8, rows_per_group: int = 30_000) -> pd.DataFrame:
    path = CLEANED_DATA_PATH / "car_data.parquet"
    parquet_file = pq.ParquetFile(path)
    frames = []
    columns = [column for column in ["session_key", "driver_number", "Speed", "Throttle", "Brake", "DRS", "lap_number"] if column in parquet_file.schema.names]
    step = max(parquet_file.num_row_groups // max_row_groups, 1)
    for row_group_idx in range(0, parquet_file.num_row_groups, step):
        if len(frames) >= max_row_groups:
            break
        chunk = parquet_file.read_row_group(row_group_idx, columns=columns).to_pandas()
        if len(chunk) > rows_per_group:
            chunk = chunk.sample(rows_per_group, random_state=42)
        frames.append(chunk)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

car_sample = sample_car_data()
speed_col = "Speed" if "Speed" in car_sample.columns else "speed"
car_sample[speed_col] = pd.to_numeric(car_sample[speed_col], errors="coerce")
car_sample = car_sample.merge(race_info[["session_key", "circuit_short_name"]], on="session_key", how="left")
circuit_speed = (
    car_sample.dropna(subset=["circuit_short_name", speed_col])
    .groupby("circuit_short_name", as_index=False)
    .agg(avg_speed_kmh=(speed_col, "mean"), p95_speed_kmh=(speed_col, lambda s: s.quantile(0.95)), samples=(speed_col, "count"))
    .sort_values("avg_speed_kmh", ascending=False)
)
if len(circuit_speed) >= 3:
    ranks = circuit_speed["avg_speed_kmh"].rank(method="first")
    circuit_speed["speed_profile"] = pd.qcut(ranks, q=3, labels=["Low-speed", "Mixed", "High-speed"])
else:
    circuit_speed["speed_profile"] = "Sampled"
circuit_speed.to_csv(OUTPUT_TABLES / "circuit_speed_profile.csv", index=False)
display(circuit_speed)

,circuit_short_name,avg_speed_kmh,p95_speed_kmh,samples,speed_profile
3,Monte Carlo,197.490471,295.0,30000,Mixed
2,Mexico City,197.490471,295.0,30000,High-speed
5,Spa-Francorchamps,197.490471,295.0,30000,High-speed
4,Sakhir,197.490471,295.0,30000,High-speed
0,Baku,173.995078,294.0,30000,Low-speed
1,Lusail,173.995078,294.0,30000,Low-speed
6,Spielberg,173.995078,294.0,30000,Low-speed
7,Suzuka,173.995078,294.0,30000,Mixed


In [12]:
fig = px.bar(
    circuit_speed.sort_values("avg_speed_kmh"),
    x="avg_speed_kmh",
    y="circuit_short_name",
    color="speed_profile",
    orientation="h",
    title="Sampled Telemetry Speed Profile by Circuit",
)
fig.write_html(OUTPUT_CHARTS / "circuit_speed_profile.html", include_plotlyjs="cdn")
fig.show()

## Final Circuit Analysis Report

In [13]:
report = {
    "notebook": NOTEBOOK_NAME,
    "timestamp": datetime.now().isoformat(),
    "circuits_analyzed": int(race_info["circuit_short_name"].nunique()),
    "sessions_analyzed": int(race_info["session_key"].nunique()),
    "highest_dnf_circuit": circuit_difficulty.iloc[0]["circuit_short_name"] if len(circuit_difficulty) else None,
    "highest_dnf_rate": float(circuit_difficulty.iloc[0]["dnf_rate"]) if len(circuit_difficulty) else 0.0,
    "highest_overtake_circuit": circuit_overtakes.iloc[0]["circuit_short_name"] if len(circuit_overtakes) else None,
    "highest_avg_overtakes": float(circuit_overtakes.iloc[0]["avg_overtakes"]) if len(circuit_overtakes) else 0.0,
    "speed_profile_circuits": int(len(circuit_speed)),
}
write_report("circuit_analysis", report)
write_insight(
    "Silver Circuit Analysis Insights",
    [
        f"Analyzed {report['circuits_analyzed']} circuits across {report['sessions_analyzed']} sessions.",
        f"Highest DNF-like circuit: {report['highest_dnf_circuit']} ({report['highest_dnf_rate']:.1%}).",
        f"Highest overtake circuit: {report['highest_overtake_circuit']} ({report['highest_avg_overtakes']:.1f} avg overtakes).",
    ],
    [],
    [
        "Use circuit DNF rate, overtake frequency, weather profile, and sampled speed profile as candidate Gold context features.",
        "Keep telemetry-derived circuit speed profile as approximate until a full feature pipeline aggregates all row groups.",
    ],
)
(CHECKPOINTS / "silver_circuit_analysis_completed.txt").write_text(json.dumps(report, indent=2), encoding="utf-8")
print(report)

{'notebook': '04_circuit_analysis', 'timestamp': '2026-06-02T00:46:17.239108', 'circuits_analyzed': 24, 'sessions_analyzed': 70, 'highest_dnf_circuit': 'Melbourne', 'highest_dnf_rate': 0.2459016393442623, 'highest_overtake_circuit': 'Sakhir', 'highest_avg_overtakes': 527.5, 'speed_profile_circuits': 8}
